In [1]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
from IPython.display import clear_output
import random
from scipy.io import loadmat
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.utils import get_lagplot
# # from useful_stuff.general_utils.RSA import  dRSA
# # from useful_stuff.general_utils.II import  dRSA, dynInformationImbalance
# from useful_stuff.image_processing.computational_models import get_relevant_output_layers
from project_specific_utils.dataloader import load_concat_regressout_meg
# from project_specific_utils.subsampling_lagged_comparisons import multivariate_lagged_comparisons
from image_processing.gaze_dep_models import save_ANN_features


In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num = 3
    neu_fs = 100
    gaze_fs = 50
    sq_side = 384
    sensors_group = 'occ'
    model_name = "vit_l_16"
    pkg = "timm"
    pseudotrial_len = 600
    pseudotrials_n = 100
    full_model_name = "alexnet_features.4"
    n_model_components = 1000
    pooling = "all"
    PCs_to_regress_out = 50
    timepts_to_regress_out = (-100, 100)
    iterations_n = 10
    repetition = 0
    signal_metric = "correlation"
    model_metric = "correlation"
    regress_out_gaze = 'PCR' # or None or pointwise
    PCs_to_regress_out = 50
    analysis_type = "RSA"
cfg = Cfg()

subjects = config["subjects"]
mod_fs = config["movie_fs"]
model_len = [round(i*cfg.neu_fs/config["movie_fs"]) for i in config["model_len"]]

In [ ]:
from project_specific_utils.dataloader import load_meg_data, load_eyetracking_data
from useful_stuff.general_utils.utils import TimeSeries, print_wise
from useful_stuff.general_utils.regression import dyn_linear_encoding
def load_concat_regressout_meg_(paths, sub_num, repetition, sensors_group, neu_fs, gaze_fs, regress_out_gaze,  PCs_to_regress_out, timepts_to_regress_out=(-100,100), rank=0):
    neu = []
    print_wise(f"Loading MEG signal: {regress_out_gaze=}", rank)
    runs = np.arange(1,4)+3*repetition 
    nans_rows = []
    for idx, i_run in enumerate(runs):
        run_neu, labels = load_meg_data(paths, sub_num, i_run, sensors_group, neu_fs)
        if np.any(np.isnan(run_neu)):
            print_wise(i_run, "contains nan")
        run_neu.z_score_feats()
        model_len = round(config["model_len"][idx]*neu_fs/config["movie_fs"])
        run_neu = TimeSeries(run_neu[:model_len], run_neu.get_fs())
        # checks if there is any sensor that has been discarded
        nan_positions = np.argwhere(np.isnan(run_neu.array))
        if nan_positions.size > 0:
            print(f"Total NaNs: {nan_positions.shape[0]}")
        rows_with_nan = np.unique(nan_positions[:, 0])
        if rows_with_nan.size > 0:
            for bad_row in rows_with_nan:
                if np.all(np.isnan(run_neu.array[bad_row, :])):  
                    nans_rows.append(bad_row)
                else:
                    raise ValueError(f"Not all elements in {bad_row} are NaNs")
                # end
        if idx == 1: # i.e. if it is the 2nd part of the movie (run 2 or run 5)
            run_neu = run_neu[3*neu_fs:]
        neu.append(run_neu[:])
    # end for i_run in range(1,4):
    for idx, run in enumerate(neu):
        neu[idx] = np.delete(run, nans_rows, axis=0)

    for idx, run_neu in enumerate(neu):
        if regress_out_gaze:
            run_gaze, _ = load_eyetracking_data(paths, sub_num, idx +1+ repetition*3, gaze_fs, xy=True)
            run_gaze.z_score_feats()
            run_gaze.resample(run_gaze.get_fs())
            run_gaze = TimeSeries(run_gaze[:model_len], run_gaze.get_fs())
            run_neu = TimeSeries(run_neu, neu_fs)
            dyn_regr_obj = dyn_linear_encoding('lr', 'same', None)
            if regress_out_gaze == "PCR":
                run_neu, _ = dyn_regr_obj.delay_embed_PCR_regress_out(run_gaze, run_neu, timepts_to_regress_out, PCs_to_keep=PCs_to_regress_out, pad_mode='edge', crop_end=True)
            elif regress_out_gaze == "lag0":
                run_neu = dyn_regr_obj.pointwise_regress_out(run_gaze, run_neu, regression_type=None) 
            # end if cfg.regr_out_eyes == "PCR":
        neu[idx] = run_neu[:]
    # end if cfg.regr_out_eyes:
    len_runs = [i.shape for i in neu]
    print_wise(f"Shape runs {runs}: {len_runs}", rank=rank)
    neu = np.concatenate(neu, axis=1)
    neu = TimeSeries(neu, neu_fs)
    return neu
# EOF



In [6]:
from scipy.io import savemat
repetitions = [0,1]
rank = 0
for i_sub in [11,]:
    for i_rep in repetitions[:1]:
        n = load_concat_regressout_meg_(paths, i_sub, i_rep, cfg.sensors_group, cfg.neu_fs, cfg.gaze_fs, cfg.regress_out_gaze, cfg.PCs_to_regress_out, timepts_to_regress_out=cfg.timepts_to_regress_out, rank=0)
        assert not np.any(np.isnan(n.array))

08:27:27 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
08:27:32 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]


In [7]:
for i_sub in [11,]:
    for i_rep in repetitions[:1]:
        n_old = load_concat_regressout_meg_(paths, i_sub, i_rep, cfg.sensors_group, cfg.neu_fs, cfg.gaze_fs, cfg.regress_out_gaze, cfg.PCs_to_regress_out, timepts_to_regress_out=cfg.timepts_to_regress_out, rank=0)

08:28:01 - rank 0 Loading MEG signal: regress_out_gaze='PCR'
08:28:05 - rank 0 Shape runs [1 2 3]: [(41, 87379), (41, 86278), (41, 79071)]


In [41]:
run_neu, labels = load_meg_data(paths, 12, 5, "occ", 100)

In [ ]:
nan_positions = np.argwhere(np.isnan(run_neu.array))
if nan_positions.size > 0:
    print(f"Total NaNs: {nan_positions.shape[0]}")
rows_with_nan = np.unique(nan_positions[:, 0])
if rows_with_nan.size > 0:
    for bad_row in rows_with_nan:
        if np.all(np.isnan(run_neu.array[bad_row, :])):
            run_neu.array = np.delete(run_neu.array, bad_row, axis=0)
            nans_rows.append(bad_row)
        else:
            raise ValueError(f"Not all elements in {bad_row} are NaNs")
        # end
    


10